# PlanGen — Tier-2 Placer Training (resumable, multi-account)

**First:** `Runtime -> Change runtime type -> T4 GPU -> Save`.

Run cells top to bottom. Training saves a portable `checkpoint.pt` to
your Drive **after every epoch**, so a disconnect never loses progress.

### Switching Colab accounts (when one hits its GPU limit)
1. On the **old** account: run **Cell 8** to download `checkpoint.pt`.
2. On the **new** account: run Cells 1–3, then **Cell 4** to upload
   that `checkpoint.pt`, then skip to **Cell 6** — it resumes
   automatically from where the last account stopped.


In [ ]:
# 1. GPU check  --  must print CUDA: True
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '<< NO GPU: Runtime -> GPU >>')


In [ ]:
# 2. Upload the bundle  --  click 'Choose Files' and pick plangen_colab.zip
from google.colab import files
files.upload()


In [ ]:
# 3. Unzip, install deps, mount Drive
!unzip -q -o plangen_colab.zip -d /content/
!pip -q install ijson scipy tqdm
import sys, os
sys.path.insert(0, '/content/plangen_remastered')
from google.colab import drive; drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/plangen_placer'
os.makedirs(CKPT, exist_ok=True)
print('code + data ready.  checkpoint dir ->', CKPT)
print('existing checkpoint?', os.path.exists(os.path.join(CKPT, 'checkpoint.pt')))


### Cell 4 — only when CONTINUING on a NEW account
Upload the `checkpoint.pt` you downloaded from the previous account.
**Skip this cell on your first account / first run.**


In [ ]:
# 4. (new account only) upload checkpoint.pt from the previous account
from google.colab import files
import shutil, os
up = files.upload()                       # pick checkpoint.pt
for name in up:
    if name.endswith('checkpoint.pt'):
        shutil.copy(name, os.path.join(CKPT, 'checkpoint.pt'))
        print('installed ->', os.path.join(CKPT, 'checkpoint.pt'))


In [ ]:
# 5. Sanity check (~30s) -- proves the pipeline runs before the long job
import sys, importlib, tier2_placer.train as T; importlib.reload(T)
sys.argv = ['train', '--smoke', '--out-dir', '/content/smoke', '--device', 'cuda']
T.main()


In [ ]:
# 6. TRAINING  (resumable). Re-run this SAME cell after any disconnect or
#    on a new account (after Cell 4) -- --resume continues automatically.
#    A live per-step progress bar shows speed + ETA for each epoch.
import sys, importlib, tier2_placer.train as T; importlib.reload(T)
sys.argv = ['train',
    '--epochs', '60', '--batch', '16', '--lr', '3e-4',
    '--eval-every', '2', '--eval-n', '50',
    '--out-dir', CKPT, '--device', 'cuda', '--resume']
T.main()


### Cell 7 — save your progress to move accounts
`checkpoint.pt` is already on your Drive (saved every epoch). Run this to
pull it down so you can upload it on the next account (Cell 4).


In [ ]:
# 7. Download checkpoint.pt (carry to the next account) + current best weights
from google.colab import files
import os
files.download(os.path.join(CKPT, 'checkpoint.pt'))
if os.path.exists(os.path.join(CKPT, 'placer.npz')):
    files.download(os.path.join(CKPT, 'placer.npz'))


In [ ]:
# 8. MERGE GATE (run once training is fully done) -- trained vs baseline.
#    This printout is the number that decides whether it ships.
import numpy as np, os
from tier2_placer.numpy_infer import NumpyPlacer
from tier2_placer.tier2_placer import Tier2Placer
from engine.orchestrator import Orchestrator
from harness.briefs import golden_briefs

placer = Tier2Placer(NumpyPlacer.from_npz(os.path.join(CKPT, 'placer.npz')))
briefs = list(golden_briefs())

def scores(orch):
    out = []
    for b in briefs:
        r = orch.generate(b)
        out.append(r.best.verdict.soft_score if r.best else 0.0)
    return np.array(out, dtype=float)

base  = scores(Orchestrator())
tuned = scores(Orchestrator(proposer=placer))
delta = tuned - base
print(f'PriorProposer {base.mean():6.2f}   Tier2Placer {tuned.mean():6.2f}   '
      f'delta {delta.mean():+.2f}')
print(f'worst single brief: {delta.min():+.2f}  (gate allows no worse than -5)')
ok = tuned.mean() > base.mean() and delta.min() > -5.0
print('VERDICT:', 'PASS - merge it' if ok else 'FAIL - stays on branch')
for b, x, y in zip(briefs, base, tuned):
    print(f'  {getattr(b, "name", "brief"):<28} {x:6.2f} -> {y:6.2f}  {y-x:+6.2f}')


In [ ]:
# 9. Final download -- send me placer.npz + the Cell 8 printout
from google.colab import files
files.download(os.path.join(CKPT, 'placer.npz'))
